# Nifty50 Multimodal Transformer Demo

This notebook demonstrates a leakage-safe multimodal pipeline for short-horizon Nifty50 stock outperformance prediction. The project combines four inputs for each stock-date sample: tabular OHLCV features, FinBERT-style text embeddings, GAF/MTF price-image embeddings, and a 37-feature relational KG vector built from sector, peer, and regime context. The results shown here anchor on Run C: a 6-stock training universe, a 49-ticker OHLCV peer universe, 1,260 aligned samples, purged walk-forward CV, and the corrected daily-aggregated backtest.


## What This Notebook Covers

- Loads a committed Run C demo subset; no yfinance calls, no FinBERT download, and no training.
- Visualizes each modality from the same aligned artifact.
- Shows the fusion model structure and the saved checkpoint shape.
- Reports Run C ablation, modality-independence, and corrected backtest numbers.
- Uses ablation-derived modality contribution as the interpretability view because the saved model uses mean pooling and does not expose attention weights.


## Setup

Setup installs the local package when needed and loads cached files from `notebooks/colab/demo_data`. The data is already committed with the notebook, so the demonstration does not fetch market data or model embeddings from external services.


In [ ]:
%%capture setup_log
import os
import sys
from pathlib import Path

# When opened directly in Colab, clone the repository once. When run from a
# local checkout, this block leaves the working directory unchanged.
REPO_URL = "https://github.com/Randhir123/nifty50-multimodal-transformer.git"
if not Path("notebooks/colab/demo_data").exists():
    if not Path("nifty50-multimodal-transformer").exists():
        !git clone -q {REPO_URL}
    %cd nifty50-multimodal-transformer

!{sys.executable} -m pip install -q -e .


In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("default")
MODALITY_COLORS = {
    "tabular": "#2F5597",
    "text": "#70AD47",
    "image": "#C55A11",
    "kg": "#8064A2",
}

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "notebooks" / "colab" / "demo_data"
assert DATA_DIR.exists(), f"Demo data not found at {DATA_DIR}"

arrays = np.load(DATA_DIR / "multimodal_samples_demo.npz", allow_pickle=False)
predictions = pd.read_csv(DATA_DIR / "predictions_demo.csv", parse_dates=["end_date"])
summary = json.loads((DATA_DIR / "run_c_summary.json").read_text())
feature_names = json.loads((DATA_DIR / "tabular_feature_names.json").read_text())
kg_feature_names = json.loads((DATA_DIR / "kg_feature_names.json").read_text())
news_samples = pd.read_json(DATA_DIR / "news_samples_demo.json")

print(f"Demo samples: {arrays['y'].shape[0]}")
print("Tabular:", arrays["tabular_tokens"].shape)
print("Text:   ", arrays["text_tokens"].shape)
print("Image:  ", arrays["image_tokens"].shape)
print("KG:     ", arrays["kg_tokens"].shape)
print("Date range:", str(pd.to_datetime(arrays["end_dates"]).min().date()), "to", str(pd.to_datetime(arrays["end_dates"]).max().date()))


## Modality 1: Tabular OHLCV Windows

The tabular branch sees a 20-day rolling window with 11 leakage-safe market features. These encode short-horizon return, realized volatility, intraday range, moving-average distance, relative volume, and stock-minus-index movement. Every row ends at the sample date, so future prices never enter the feature window.


In [ ]:
sample_idx = 0
stock = str(arrays["stock_ids"][sample_idx])
end_date = pd.Timestamp(arrays["end_dates"][sample_idx]).date()
tab = pd.DataFrame(arrays["tabular_tokens"][sample_idx], columns=feature_names)
print(f"Sample: {stock} ending {end_date}")
display(tab.tail(6).round(4))

fig, ax1 = plt.subplots(figsize=(8, 4))
x = np.arange(len(tab))
ax1.plot(x, tab["cum_return_3d"], label="3d cumulative return", color=MODALITY_COLORS["tabular"])
ax1.plot(x, tab["cum_return_10d"], label="10d cumulative return", color="#7F7F7F")
ax1.set_xlabel("Window day")
ax1.set_ylabel("Return feature")
ax1.legend(loc="upper left")
ax2 = ax1.twinx()
ax2.plot(x, tab["realized_vol_10d"], label="10d realized vol", color="#A64D79", linestyle="--")
ax2.set_ylabel("Volatility")
ax2.legend(loc="upper right")
plt.title("Tabular window for one stock-date sample")
plt.tight_layout()
plt.show()


*Caption: the model receives the whole 20-day sequence, not a single end-of-window row. The plotted features are representative slices of the 11-dimensional tabular token stream.*


## Modality 2: Text Embeddings

The text branch uses cached 768-dimensional vectors aligned to each stock-date. In the full pipeline these are built from as-of-date company text, so only records dated on or before the sample date can influence the embedding. The demo also ships a few raw market-summary headlines so the encoded modality has something human-readable beside it.


In [ ]:
stock_news = news_samples[news_samples["stock_id"] == stock].head(5)
display(stock_news[["stock_id", "event_date", "title", "body_text"]])

text_tokens = arrays["text_tokens"]
text_frame = pd.DataFrame({
    "stock_id": arrays["stock_ids"].astype(str),
    "x": text_tokens[:, 0],
    "y": text_tokens[:, 1],
    "activation_mean": text_tokens.mean(axis=1),
})
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(text_tokens[sample_idx], bins=30, color=MODALITY_COLORS["text"], alpha=0.85)
axes[0].set_title("One 768-d text vector")
axes[0].set_xlabel("Activation")
axes[0].set_ylabel("Count")
for sid, group in text_frame.groupby("stock_id"):
    axes[1].scatter(group["x"], group["y"], s=28, label=sid, alpha=0.75)
axes[1].set_title("First two embedding dimensions")
axes[1].set_xlabel("dim 0")
axes[1].set_ylabel("dim 1")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


*Caption: this is not a semantic map; it is a compact diagnostic view showing that cached text vectors are present, finite, and aligned by ticker.*


## Modality 3: Price Images

The image branch encodes each 20-day close-price path as Gramian Angular Field and Markov Transition Field style images, then compresses the two-channel representation with a small CNN. GAF preserves temporal geometry of the scaled price path; MTF summarizes transition structure between quantized price states. Both are built from prices ending at the sample date.


In [ ]:
def ticker_to_raw_name(ticker):
    return ticker.replace(".", "_").replace("-", "_") + ".csv"

def scaled_close_for_sample(ticker, end_date, window=20):
    raw = pd.read_csv(DATA_DIR / "raw" / ticker_to_raw_name(ticker), parse_dates=["date"])
    raw = raw.sort_values("date")
    hist = raw[raw["date"] <= pd.Timestamp(end_date)].tail(window)
    return hist["date"], hist["close"].to_numpy(dtype=float)

def simple_gaf(values, image_size=32):
    v = np.asarray(values, dtype=float)
    scaled = 2 * (v - v.min()) / (v.max() - v.min() + 1e-8) - 1
    phi = np.arccos(np.clip(scaled, -1, 1))
    gaf = np.cos(phi[:, None] + phi[None, :])
    idx = np.linspace(0, len(v) - 1, image_size).astype(int)
    return gaf[np.ix_(idx, idx)]

def simple_mtf(values, image_size=32, bins=8):
    v = np.asarray(values, dtype=float)
    quantiles = np.quantile(v, np.linspace(0, 1, bins + 1))
    states = np.clip(np.digitize(v, quantiles[1:-1]), 0, bins - 1)
    trans = np.zeros((bins, bins), dtype=float)
    for a, b in zip(states[:-1], states[1:]):
        trans[a, b] += 1
    trans = trans / (trans.sum(axis=1, keepdims=True) + 1e-8)
    field = trans[states[:, None], states[None, :]]
    idx = np.linspace(0, len(v) - 1, image_size).astype(int)
    return field[np.ix_(idx, idx)]

dates, close = scaled_close_for_sample(stock, end_date)
gaf = simple_gaf(close)
mtf = simple_mtf(close)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].plot(dates, close, color=MODALITY_COLORS["image"])
axes[0].set_title("Close-price window")
axes[0].tick_params(axis="x", rotation=45)
axes[1].imshow(gaf, cmap="RdBu_r", aspect="auto")
axes[1].set_title("GAF channel")
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[2].imshow(mtf, cmap="viridis", aspect="auto")
axes[2].set_title("MTF channel")
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.tight_layout()
plt.show()
print("Cached CNN image token shape for this sample:", arrays["image_tokens"][sample_idx].shape)


*Caption: the notebook reconstructs the visual encodings for inspection; the model itself consumes the cached 16-dimensional CNN output stored in the artifact.*


## Modality 4: Relational KG Features

The KG modality is a 37-feature vector rather than a graph database. It encodes sector membership, sector-index behavior, peer correlations, rank and z-score features within sector, lead-lag relations, and Nifty50 regime context. Run C computes these features for the 6 training stocks using a 49-ticker OHLCV peer universe, while preserving the date cutoff for each sample.


In [ ]:
kg = pd.Series(arrays["kg_tokens"][sample_idx], index=kg_feature_names)
groups = {
    "Sector context": kg.iloc[:12],
    "Peer relationships": kg.iloc[12:27],
    "Market regime": kg.iloc[27:],
}
for name, values in groups.items():
    print(f"\n{name}")
    display(values.to_frame("value").T.round(4))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, values) in zip(axes, groups.items()):
    values.plot(kind="barh", ax=ax, color=MODALITY_COLORS["kg"])
    ax.set_title(name)
    ax.set_xlabel("Value")
plt.tight_layout()
plt.show()


*Caption: the first eight KG entries are sector one-hot values; the remaining entries are continuous peer, sector, and benchmark context computed as of the sample date.*


## Architecture

```mermaid
flowchart LR
    T["20 x 11 tabular window"] --> TP["Linear projection"]
    I["16-d image embedding"] --> IP["Linear projection"]
    X["768-d text embedding"] --> XP["Linear projection"]
    K["37-d KG vector"] --> KP["Linear projection"]
    TP --> F["Transformer encoder fusion"]
    IP --> F
    XP --> F
    KP --> F
    F --> M["Mean pooling"]
    M --> C["Binary outperformance logit"]
```

Each modality is projected to the same `model_dim`, tagged with a modality embedding, concatenated into one token sequence, and passed through a small Transformer encoder. Run C uses mean pooling, not CLS pooling.


In [ ]:
from src.models.fusion import FusionTransformer, FusionTransformerConfig

checkpoint = torch.load(DATA_DIR / "model_checkpoint.pt", map_location="cpu", weights_only=False)
config = FusionTransformerConfig(**checkpoint["config"])
model = FusionTransformer(config)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

rows = []
for name, param in model.named_parameters():
    top = name.split(".")[0]
    rows.append({"module": top, "parameter": name, "count": int(param.numel())})
param_df = pd.DataFrame(rows)
module_counts = param_df.groupby("module", as_index=False)["count"].sum().sort_values("count", ascending=False)
display(module_counts)
print(f"Total parameters: {param_df['count'].sum():,}")
print(f"model_dim={config.model_dim}, heads={config.num_heads}, layers={config.num_layers}, ff_dim={config.ff_dim}, pooling={config.pooling}")


## Cross-Validation and Leakage Safety

Run C uses purged walk-forward CV with an embargo between train and validation periods. The purpose is simple: samples near a fold boundary can share overlapping forward-return horizons, so the split leaves a gap before the validation window. The repository also contains integration tests that check date cutoffs for text, KG, and backtest-time prediction handling.


In [ ]:
folds = []
full_pred = pd.read_csv(DATA_DIR / "predictions_demo.csv", parse_dates=["end_date"])
# The demo subset is not the full CV table, so this is an orientation view by date thirds.
dates = np.array(sorted(full_pred["end_date"].dt.normalize().unique()))
for fold, chunk in enumerate(np.array_split(dates, 3)):
    folds.append({"fold": fold, "demo_start": pd.Timestamp(chunk[0]).date(), "demo_end": pd.Timestamp(chunk[-1]).date(), "samples": int(full_pred[full_pred["end_date"].isin(chunk)].shape[0])})
display(pd.DataFrame(folds))

fig, ax = plt.subplots(figsize=(8, 2.6))
for row in folds:
    ax.barh(row["fold"], row["samples"], color="#9E9E9E")
    ax.text(row["samples"] + 1, row["fold"], f"{row['demo_start']} to {row['demo_end']}", va="center")
ax.set_xlabel("Demo samples")
ax.set_ylabel("Fold-like date block")
ax.set_title("Chronological validation orientation")
plt.tight_layout()
plt.show()


## Training Characteristics

The run used BCE-with-logits loss, fixed optimizer settings, and a small fusion model. The saved checkpoints contain final validation diagnostics per fold, not full per-epoch training curves, so this cell plots the stored fold-level checkpoint metrics rather than reconstructing a history that was not saved.


In [ ]:
history = pd.read_csv(DATA_DIR / "training_history_demo.csv")
display(history.round(4))
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(history["fold"], history["val_loss"], marker="o", color="#595959", label="validation loss")
ax1.set_xlabel("Fold")
ax1.set_ylabel("Validation loss")
ax2 = ax1.twinx()
ax2.plot(history["fold"], history["val_roc_auc"], marker="o", color=MODALITY_COLORS["text"], label="validation ROC-AUC")
ax2.set_ylabel("ROC-AUC")
ax1.set_title("Saved checkpoint diagnostics")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()


## Results: Run C

Run C is the canonical result for this notebook: 6 training tickers, a 49-ticker OHLCV peer universe for KG features, 1,260 samples, three CV folds, and the corrected backtest that aggregates overlapping holdings by daily portfolio return.


In [ ]:
ablation = pd.DataFrame(summary["ablation_results"])
ablation["roc_auc_label"] = ablation.apply(lambda r: f"{r.roc_auc_mean:.3f} ? {r.roc_auc_std:.3f}", axis=1)
display(ablation[["variant", "roc_auc_label", "accuracy_mean", "f1_mean"]].round(4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ablation["variant"], ablation["roc_auc_mean"], xerr=ablation["roc_auc_std"], color="#6B8EBD")
ax.axvline(0.5, color="#444444", linestyle="--", linewidth=1)
ax.set_xlabel("ROC-AUC mean ? std")
ax.set_title("Run C ablation results")
plt.tight_layout()
plt.show()


The strongest single additions in Run C are text and image. KG improves over tabular-only, but modestly. The all-four fusion is below the best single-modality additions, which is a scale and fusion limitation rather than evidence that the extra modalities are useless.


In [ ]:
def demo_equity_curve(pred_df, top_k=3):
    daily = []
    for date, group in pred_df.groupby(pred_df["end_date"].dt.normalize()):
        selected = group.sort_values("predicted_score", ascending=False).head(top_k)
        daily.append({
            "date": date,
            "model_return": selected["future_return"].mean(),
            "benchmark_return": group["benchmark_return"].mean(),
        })
    out = pd.DataFrame(daily).sort_values("date")
    out["model_equity"] = (1 + out["model_return"]).cumprod()
    out["benchmark_equity"] = (1 + out["benchmark_return"]).cumprod()
    return out

curve = demo_equity_curve(predictions)
metrics = summary["backtest"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["date"], curve["model_equity"], label="Demo model picks", color=MODALITY_COLORS["text"])
ax.plot(curve["date"], curve["benchmark_equity"], label="Demo benchmark", color="#7F7F7F")
ax.set_title("Cached demo equity curve")
ax.set_ylabel("Growth of 1.0")
ax.legend()
plt.tight_layout()
plt.show()
print("Run C corrected backtest headline:")
print(f"model return={metrics['total_return_model']:.2%}, benchmark={metrics['total_return_benchmark']:.2%}, Sharpe={metrics['sharpe_ratio']:.2f}, max drawdown={metrics['max_drawdown_model']:.2%}")


## Per-Prediction Interpretability Fallback

The requested attention-attribution view is not mechanically available from the saved model: Run C uses mean pooling and the current `FusionTransformer` returns logits only, not attention weights. Rather than report misleading attention numbers, this notebook uses a conservative fallback: modality contribution estimated from ablation deltas against the tabular-only baseline, plus individual prediction score cards.


In [ ]:
base = float(ablation.loc[ablation["variant"] == "tabular_only", "roc_auc_mean"].iloc[0])
contrib = pd.DataFrame({
    "modality": ["kg", "image", "text", "all four"],
    "roc_auc_delta_vs_tabular": [
        float(ablation.loc[ablation["variant"] == "tabular_kg", "roc_auc_mean"].iloc[0]) - base,
        float(ablation.loc[ablation["variant"] == "tabular_image", "roc_auc_mean"].iloc[0]) - base,
        float(ablation.loc[ablation["variant"] == "tabular_text", "roc_auc_mean"].iloc[0]) - base,
        float(ablation.loc[ablation["variant"] == "tabular_image_text_kg", "roc_auc_mean"].iloc[0]) - base,
    ],
})
colors = [MODALITY_COLORS.get(m, "#666666") for m in contrib["modality"]]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(contrib["modality"], contrib["roc_auc_delta_vs_tabular"], color=colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_xlabel("ROC-AUC delta vs tabular-only")
ax.set_title("Ablation-derived modality contribution")
plt.tight_layout()
plt.show()
display(contrib.round(4))


In [ ]:
examples = pd.concat([
    predictions.assign(correct=(predictions["predicted_score"] >= 0.5).astype(int) == predictions["y_true"]).query("correct").sort_values("predicted_score", ascending=False).head(1),
    predictions.assign(correct=(predictions["predicted_score"] >= 0.5).astype(int) == predictions["y_true"]).query("correct").sort_values("predicted_score", ascending=True).head(1),
    predictions.assign(correct=(predictions["predicted_score"] >= 0.5).astype(int) == predictions["y_true"]).query("not correct").sort_values("predicted_score", ascending=False).head(1),
    predictions.assign(correct=(predictions["predicted_score"] >= 0.5).astype(int) == predictions["y_true"]).query("not correct").sort_values("predicted_score", ascending=True).head(1),
]).drop_duplicates().head(4)

display(examples[["stock_id", "end_date", "predicted_score", "y_true", "future_return"]].round(4))
fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{r.stock_id}\n{r.end_date.date()}" for r in examples.itertuples()]
ax.bar(labels, examples["predicted_score"], color="#4F81BD")
ax.axhline(0.5, color="#333333", linestyle="--", linewidth=1)
ax.set_ylabel("Predicted probability")
ax.set_title("Representative cached predictions")
plt.tight_layout()
plt.show()


*Caption: the contribution plot is global, not per-sample attention. It shows what changed when each modality was available during CV. The score cards then ground that global result in individual cached predictions.*


## Limitations

- The training universe is still small: six stocks, even though KG features use a broader OHLCV peer universe.
- Results are single-seed and should not be read as a stable production trading claim.
- All-four fusion underperforms the best single-modality additions in Run C, which points to limited data scale or fusion capacity.
- The backtest is corrected for overlapping holding periods, but transaction costs, slippage, and deployment constraints remain outside this demo.


## Next Steps

For implementation details, read the repository README, `docs/findings.md`, and `docs/design-notes.md`. To run a new experiment on Colab compute, use `notebooks/colab/run_experiment.ipynb`; this notebook is only the cached demonstration path.
